# Klaatch Audio and Text Analysis: Predicting Loneliness from Multimodal Features

## Overview

This notebook analyzes the Klaatch dataset (2021-2023) to predict loneliness (CEL scores) using multiple feature modalities:
- **Audio Features**: Whisper embeddings, OpenSmile acoustic features, Librosa features
- **Text Features**: LIWC linguistic patterns, word categories, word embeddings
- **Demographic Features**: Age, gender, race

## Methodology

1. **Propensity Score Matching**: Create balanced demographic subgroups for fair model comparison
2. **Feature Extraction**: Extract multimodal features from audio recordings and transcripts
3. **Predictive Modeling**: Train multiple ML models (Ridge, SVR, ExtraTrees, XGBoost) using DLATK
4. **Fairness Analysis**: Test model performance across demographic subgroups
5. **Interpretation**: Analyze top predictive features and topics

## Key Research Questions

- Can audio features predict loneliness as effectively as text features?
- Do models perform fairly across gender and racial subgroups?
- What linguistic and acoustic patterns are most predictive of loneliness?

## Tools & Libraries

- **DLATK**: Differential Language Analysis ToolKit for feature extraction and prediction
- **scikit-learn**: Machine learning models and propensity score matching
- **MySQL**: Database for storing features and prediction results
- **Pandas/NumPy**: Data processing and analysis

## Demographic Subgroup Creation

Split data by gender for propensity score matching analysis. This creates balanced male and female subsets.

## Dataset Information

**Source**: Klaatch audio recordings with transcripts (2021-2023)  
**Target Variable**: CEL (Cumulative Emotion Loneliness) Total Score  
**CEL Components**:
- CELVAL1: Emotional loneliness component
- CELVAL2: Social loneliness component  
- CELVAL3: Existential loneliness component

**Features**:
- Audio recordings (.mp3 files)
- Text transcripts (cleaned and preprocessed)
- Demographics: Age, Gender, Race
- Multiple audio feature representations (Whisper, OpenSmile, Librosa)
- Text feature representations (LIWC, categories, embeddings)

## Notebook Structure

1. **Setup & Data Loading** (Cells 3-10): Import libraries, load data, clean text
2. **Preprocessing** (Cells 11-32): Propensity score matching, demographic subgroups
3. **DLATK Predictions** (Cells 33-73): Ridge regression baseline across all features
4. **SVR Models** (Cells 74-116): Support Vector Regression predictions
5. **ExtraTrees Models** (Cells 117-174): Extra Trees Regressor predictions
6. **XGBoost Models** (Cells 175-185): XGBoost gradient boosting predictions
7. **Binary Classification** (Cells 186-191): Loneliness as binary outcome
8. **Topic Analysis** (Cells 192+): Interpret top predictive features

---

## Race Classification

Define race categories for demographic analysis. Grouping White race variants together for consistent categorization.

## Feature Selection and Word Count

Select relevant columns for analysis and add word count feature for filtering short messages.

## Text Data Cleaning

Remove unwanted patterns from text data including:
- Likert scale terms (strongly agree, disagree, neutral, etc.)
- Timestamps in format [HH:MM:SS]
- Numbers and special characters
- Speaker identifiers
- Common stop words that don't add meaningful information

This cleaning ensures the text analysis focuses on substantive content.

## Data Loading

Load the Klaatch dataset from CSV and perform initial data cleaning.

## Import Libraries

Load all required Python libraries for data processing, machine learning, database connectivity, and propensity score matching.

# Klaatch DLATK Prediction Analysis

This notebook performs propensity score matching and predictive modeling analysis on the Klaatch audio dataset using DLATK (Differential Language Analysis ToolKit).

## Overview

The analysis consists of the following main components:
1. **Data Loading and Preprocessing** - Load the Klaatch dataset and clean text data
2. **Propensity Score Matching** - Create matched demographic pairs to control for confounding variables
3. **Database Setup** - Configure MySQL database for DLATK feature extraction
4. **DLATK Prediction Models** - Run multiple regression models (Ridge, SVR, ExtraTrees, XGBoost) on various audio and linguistic features
5. **Topic Analysis** - Extract and analyze top topics from the data

## Dataset
- Source: Klaatch final data CSV
- Target variables: CEL Total, CELVAL1, CELVAL2, CELVAL3
- Demographics: Gender, Race, Age

In [101]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
#from sklearn.ensemble import VotingRegressor
import re
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import sqlalchemy
import mysql.connector

In [85]:
df = pd.read_csv("/home/vinmayk/Klaatch_final_data.csv")
df.drop('Unnamed: 0', axis=1, inplace=True)
df.rename(columns={'New ID_x': 'New ID'}, inplace=True)
df.rename(columns={'Old ID_x': 'Old ID'}, inplace=True)
df.shape

(1465, 28)

In [86]:
import pandas as pd
import re

# Assuming merged_df is already defined as per your initial code

# Define the patterns to remove
patterns_to_remove = [
    'strongly agree',
    'strongly disagree',
    'strongly_agree',
    'strongly_disagree',
    'agree\w*',          # Matches agree, agreed, agreement, agreeable, agreeing
    'disagree\w*',       # Matches disagree, disagreed, disagreement, disagreeing
    'neutral',
    'Speaker',
    'strongly agree\w*', # Matches strongly agree variations
    'strongly disagree\w*',
    'strongly_agree\w*',
    'strongly_disagree\w*',
    'Speaker\w*',
    'speaker\w*',
    'nn\w*',
    'nnspeaker\w*',
    '!', '\$', ',', '\.', '\.\.', '\.\s*mm hmm', '\.\.',
    '0', '00 \]', '0:00', '0:00:00',
    '1', '10', '100', '11', '12', '13', '14', '15', '16', '18',
    '2', '20', '20 years', '25',
    '3', '30', '4', '40',
    '5', '5 to 10', '50', '90',
    ':', ': 00 \]', '\?', '\]', '\] hello', '\] hello \.',
    "don't\w*", 'not\w*', 'strongly\w*', 'definitely\w*', 'neither\w*', 'lisa\w*'
]



# Regex pattern to identify timestamps like [00:24:00]
timestamp_pattern = r'\[\d{2}:\d{2}:\d{2}\]'

# Note: Do NOT escape patterns that contain regex operators like \w
escaped_patterns = []
for pattern in patterns_to_remove:
    if '\\w' in pattern:
        escaped_patterns.append(pattern)  # Keep regex operators
    else:
        escaped_patterns.append(re.escape(pattern))  # Escape others

words_regex = r'\b(?:' + '|'.join(escaped_patterns) + r')\b'


# Combine word/phrase patterns with timestamp pattern
full_regex_pattern = f'(?:{words_regex})|(?:{timestamp_pattern})'

# Remove the patterns from the 'Text' column
df['Text'] =  df['Text'].str.replace(
    full_regex_pattern, 
    '', 
    regex=True, 
    flags=re.IGNORECASE
)

# Clean up any extra whitespace
df['Text'] = df['Text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Verify the changes
#print(stopwords_merged_df[['Filename', 'Text']].head())
df

,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Old ID_y,Age,...,Monthly Income,Cognitive Impaired,Dementia,Filepath,Cleaned Name,Old ID_y.1,Date,Original_Filename,New ID_y,Text_from_other
0,NaN,559.0,559_2021-01-22,"Hello. OK. Sure. Yesterday, I went to an appoi...",5,1,0,4,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/559_01-22-2021...,559_01-22-2021_11-40am,559.0,2021-01-22,559_01-22-2021_11-40am_wb.mp3,NaN,"Hello. OK. Sure. Yesterday, I went to an appoi..."
1,938730.0,66.0,66_2021-01-26,Hello. Please leave a message after the tone. ...,3,1,1,1,66.0,95.0,...,1841.96,NaN,YES,/sandata/karthik9/nwa/New_audio/66_01-26-2021_...,66_01-26-2021_15-20min,66.0,2021-01-26,66_01-26-2021_15-20min_wb.mp3,938730.0,Hello. Please leave a message after the tone. ...
2,938936.0,340.0,340_2021-01-26,"Yes. I'm fine, how are you? Ok. Well, what I u...",2,1,0,1,340.0,82.0,...,1630.00,NaN,NaN,/sandata/karthik9/nwa/New_audio/340_01-26-2021...,340_01-26-2021_15-05min,340.0,2021-01-26,340_01-26-2021_15-05min_wb.mp3,938936.0,"Yes. I'm fine, how are you? Ok. Well, what I u..."
3,NaN,343.0,343_2021-01-26,"Hello. All right. I'll do it now. Sure. No, no...",1,1,0,0,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/343_01-26-2021...,343_01-26-2021_08-43min,343.0,2021-01-26,343_01-26-2021_08-43min_wb.mp3,NaN,"Hello. All right. I'll do it now. Sure. No, no..."
4,938879.0,383.0,383_2021-01-27,"Hello. Hi, I'm. Ok. OK. Yes, okay. I wake up, ...",5,1,3,1,383.0,72.0,...,885.00,NaN,NaN,/sandata/karthik9/nwa/New_audio/383_01-27-2021...,383_01-27-2021_15-35min,383.0,2021-01-27,383_01-27-2021_15-35min_wb.mp3,938879.0,"Hello. Hi, I'm. Ok. OK. Yes, okay. I wake up, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1460,NaN,155.0,155_2021-06-02,Hello. Okay. Ready. Silence. Silence. Silence....,4,3,0,1,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/155_06-02-2021...,155_06-02-2021_21-40pm,155.0,2021-06-02,155_06-02-2021_21-40pm_wb.mp3,NaN,Hello. Okay. Ready. Silence. Silence. Silence....
1461,NaN,156.0,156_2021-06-03,"Hello? How are you? Okay! So, survey. In the w...",6,1,1,4,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/156_06-03-2021...,156_06-03-2021_17-26pm,156.0,2021-06-03,156_06-03-2021_17-26pm_wb.mp3,NaN,"Hello? How are you? Okay! So, survey. In the w..."
1462,NaN,146.0,146_2021-06-04,"Hi, baby. All right. I hate surveys. Okay. All...",8,4,1,3,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/146_06-04-2021...,146_06-04-2021_15-32pm,146.0,2021-06-04,146_06-04-2021_15-32pm_wb.mp3,NaN,"Hi, baby. All right. I hate surveys. Okay. All..."
1463,NaN,128.0,128_2021-06-09,"Hello. Long time. Uh. Here line. And, uh, ques...",6,1,4,1,NaN,NaN,...,NaN,NaN,NaN,/sandata/karthik9/nwa/New_audio/128_06-09-2021...,128_06-09-2021_13-47pm,128.0,2021-06-09,128_06-09-2021_13-47pm_wb.mp3,NaN,"Hello. Long time. Uh. Here line. And, uh, ques..."


In [87]:
df = df[['New ID', 'Old ID', 'Filename', 'Text', 'CEL Total', 'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'Gender', 'Race', 'Date']].copy()

df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))

In [88]:
white_race = ['White', 'White, Jewish', 'White, Judaism', 'White,']

In [89]:
male_df = df[df['Gender']=='Male']
female_df = df[df['Gender']=='Female']
white_df = df[df['Race'].isin(white_race)]
black_df = df[df['Race']=='Black or African American']

In [75]:
gender_data = pd.concat([male_df, female_df])
race_data = pd.concat([white_df, black_df])

## Data Preprocessing: Filtering and Binary Encoding for Propensity Score Matching

### Step 1: Remove Small/Mixed Race Groups from Gender Analysis
- **Remove Asian participants**: Small sample size (9 males, 0 females) insufficient for robust comparison
- **Remove mixed-race participants**: "White, Black or African American" category creates confounding

### Step 2: Binary Encode Covariates

**For Race Comparison (`race_data`):**
- Encode Gender: Male = 0, Female = 1
- Used as covariate when comparing Black vs White participants

**For Gender Comparison (`gender_data`):**
- Encode Race: Black or African American = 0, White (all variants) = 1
- Used as covariate when comparing Male vs Female participants

### Purpose
These binary encodings prepare covariates for logistic regression models that calculate propensity scores, enabling matched pair analysis while controlling for:
- Age
- CEL Total score
- The respective demographic variable (Race or Gender)

In [76]:
gender_data = gender_data.drop(gender_data[gender_data['Race'] == 'Asian'].index)
gender_data = gender_data.drop(gender_data[gender_data['Race'] == 'White, Black or African American'].index)

race_data['Gender'] = race_data['Gender'].replace({'Male': 0}, regex=False)
race_data['Gender'] = race_data['Gender'].apply(lambda x: 1 if x != 0 else x)

gender_data['Race'] = gender_data['Race'].replace({'Black or African American': 0}, regex=False)
gender_data['Race'] = gender_data['Race'].apply(lambda x: 1 if x != 0 else x)


## Calculate Propensity Scores for Gender Comparison

### Encoding Gender as Binary
First, we encode Gender into numeric format:
- Male = 0
- Female = 1

This binary encoding is necessary for the logistic regression model.

### Building the Propensity Score Model
- **Features (X):** Race, Age, CEL Total
- **Target (y):** Gender (Male vs Female)
- **Output:** Probability of being Female given the features

### Purpose
The gender propensity score represents the likelihood of being Female (vs Male) based on:
- Race composition
- Age distribution
- CEL Total scores

We use these scores to match Male and Female participants who have similar characteristics, ensuring fair comparison by controlling for confounding variables.

In [77]:
gender_data['Gender'] = gender_data['Gender'].replace({'Male': 0, 'Female': 1}).astype(int)

gender_X = gender_data[['Race', 'Age', 'CEL Total']]
gender_y = gender_data['Gender']
gender_model = LogisticRegression(max_iter=500)
gender_data['gender_propensity_score'] = gender_model.fit(gender_X, gender_y).predict_proba(gender_X)[:, 1]

/tmp/ipykernel_2649377/3933652195.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gender_data['Gender'] = gender_data['Gender'].replace({'Male': 0, 'Female': 1}).astype(int)


## Calculate Propensity Scores for Race Comparison

### Encoding Race as Binary
First, we encode Race into numeric format:
- Black or African American = 0
- White (all variants) = 1

This binary encoding is necessary for the logistic regression model.

### Building the Propensity Score Model
- **Features (X):** Gender, Age, CEL Total
- **Target (y):** Race (Black vs White)
- **Output:** Probability of being White given the features

### Purpose
The race propensity score represents the likelihood of being White (vs Black) based on:
- Gender composition
- Age distribution  
- CEL Total scores

We use these scores to match Black and White participants who have similar characteristics, ensuring fair comparison by controlling for confounding variables.

**Note:** This corrects the previous error where we were incorrectly predicting Gender instead of Race.

In [78]:
# Create a binary Race column
race_data['Race_binary'] = race_data['Race'].map({
    'Black or African American': 0,
    'White': 1,
    'White, Jewish': 1,
    'White, Judaism': 1,
    'White,': 1
}).astype(int)

race_X = race_data[['Gender', 'Age', 'CEL Total']]
race_y = race_data['Race_binary']  # Use the encoded Race
race_model = LogisticRegression(max_iter=500)
race_data['race_propensity_score'] = race_model.fit(race_X, race_y).predict_proba(race_X)[:, 1]

In [92]:
male = gender_data[gender_data['Gender']==0]      # ✓ Male = 0
female = gender_data[gender_data['Gender']==1]    # ✓ Female = 1

white = race_data[race_data['Race'].isin(white_race)]  # ✓ Race still has strings
black = race_data[race_data['Race']=='Black or African American']  # ✓ Race still has strings



In [80]:
print(male.shape)
print(female.shape)
print(white.shape)
print(black.shape)

(397, 14)
(931, 14)
(924, 15)
(383, 15)


## Propensity Score Matching: Creating Matched Male-Female Pairs

### What This Code Does

This implements **1:1 nearest neighbor propensity score matching** to create balanced comparison groups.

### Step-by-Step Process

1. **Find Closest Matches**
   - For each Female participant, find the Male with the most similar propensity score
   - The propensity score represents similarity in Race, Age, and CEL Total
   - Add both the Female and her matched Male to the `matched_pairs` dataset

2. **Prevent Duplicate Matching**
   - Once a Male is matched, remove him from the pool
   - This ensures each Male is matched to only one Female (1:1 matching)

3. **Separate and Balance**
   - Split matched pairs back into Female and Male groups
   - Ensure equal numbers from each group (balanced dataset)

4. **Randomize**
   - Shuffle the final dataset to remove any ordering effects
   - Use `random_state=42` for reproducibility

### Result
A balanced dataset (`final_gender`) where Males and Females are matched on confounding variables:
- Similar racial distribution
- Similar age ranges
- Similar CEL Total scores

This allows fair comparison of gender differences while controlling for these factors.

In [81]:
matched_pairs = pd.DataFrame()
for _, row in female.iterrows():
    closest_match = male.iloc[(male['gender_propensity_score'] - row['gender_propensity_score']).abs().argsort()[:1]]
    matched_pairs = pd.concat([matched_pairs, row.to_frame().T, closest_match], ignore_index=True)
    male = male.drop(closest_match.index)

matched_pairs = matched_pairs.reset_index(drop=True)

female_matched = matched_pairs[matched_pairs['Gender'] == 1]  # ✓ Use 1 for Female
male_matched = matched_pairs[matched_pairs['Gender'] == 0]    # ✓ Use 0 for Male

min_entries = min(len(female_matched), len(male_matched))

female_final = female_matched.sort_values(by='gender_propensity_score').head(min_entries)
male_final = male_matched.sort_values(by='gender_propensity_score').head(min_entries)

final_gender = pd.concat([female_final, male_final]).reset_index(drop=True)

final_gender = final_gender.sample(frac=1, random_state=42).reset_index(drop=True)

final_gender

,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count,gender_propensity_score
0,938762.0,507.0,507_2022-12-06,Hello. Kelly Well. Sure. Fairly well. No. Hell...,4,1,1,2,72.0,0,0,2022-12-06,333,0.733261
1,938911.0,741.0,741_2022-08-11,"Hello.: Good. How are you? Well, I'm going to ...",3,1,1,1,86.0,0,0,2022-08-11,2201,0.849007
2,938913.0,661.0,661_2023-05-15,"Hello. Hi. Good. Yeah. Oh, yeah. Yeah. Yeah. U...",2,1,0,1,70.0,1,1,2023-05-15,461,0.544755
3,938652.0,522.0,522_2024-08-13,"000003] Hi, . Okay, how about you this time? I...",0,0,0,0,74.0,0,1,2024-08-13,2952,0.604046
4,938820.0,618.0,618_2023-04-06,"Hello. Oh, hi, Lorna. How are you? Good thing....",2,1,0,1,70.0,1,1,2023-04-06,926,0.544755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789,938885.0,509.0,509_2023-11-17,"nd now. Ha! : Oh, yeah. Yeah, I used to be. I ...",3,0,2,1,71.0,1,1,2023-11-17,764,0.551472
790,938945.0,45.0,45_2024-09-27,"ello. I know, yeah. Hello? Hey. I'm hanging in...",5,1,2,2,74.0,1,1,2024-09-27,1175,0.576977
791,938933.0,783.0,783_2023-08-02,"Hello. Hi, . How are you? Yes, I do. Sure. Oka...",3,1,1,1,78.0,1,1,2023-08-02,411,0.634884
792,981476.0,852.0,981476_2024-11-01,"Oh, I didn't know. Very good. Oh, no, no probl...",2,0,1,1,68.0,0,1,2024-11-01,829,0.520107


In [82]:
matched_pairs = pd.DataFrame()
for _, row in white.iterrows():
    closest_match = black.iloc[(black['race_propensity_score'] - row['race_propensity_score']).abs().argsort()[:1]]  # ✓ Use race_propensity_score
    matched_pairs = pd.concat([matched_pairs, row.to_frame().T, closest_match], ignore_index=True)
    black = black.drop(closest_match.index)

matched_pairs = matched_pairs.reset_index(drop=True)

white_matched = matched_pairs[matched_pairs['Race'].isin(white_race)]
black_matched = matched_pairs[matched_pairs['Race'] == 'Black or African American']

min_entries = min(len(white_matched), len(black_matched))

white_final = white_matched.sort_values(by='race_propensity_score').head(min_entries)  # ✓ Sort by race score
black_final = black_matched.sort_values(by='race_propensity_score').head(min_entries)  # ✓ Sort by race score

final_race = pd.concat([white_final, black_final]).reset_index(drop=True)

final_race = final_race.sample(frac=1, random_state=42).reset_index(drop=True)

final_race

,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count,Race_binary,race_propensity_score
0,938682.0,545.0,545_2023-04-06,"Hello. Hello. Good morning. How are you? Uh, n...",3,1,1,1,65.0,1,Black or African American,2023-04-06,790,0,0.580088
1,938695.0,687.0,687_2022-08-08,"Hello. Yet. Oh, okay. Yeah. Yeah. How you doin...",0,0,0,0,74.0,0,Black or African American,2022-08-08,1436,0,0.738187
2,938698.0,170.0,938698_2024-10-30,Hello. I'm doing pretty well. Yeah. What am I ...,2,1,1,0,87.0,1,White,2024-10-30,1206,1,0.69502
3,938917.0,173.0,173_2022-11-10,"Hello.: Oh, hi. How are you?: Good, good, good...",0,0,0,0,90.0,1,Black or African American,2022-11-10,3206,0,0.670894
4,938673.0,144.0,144_2024-07-17,"ello? Oh, okay. Um, are you recording now? Alr...",3,1,1,1,83.0,1,White,2024-07-17,3554,1,0.692447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
761,938667.0,419.0,419_2021-05-17,"Hello. You. OK, I can hear. Sure. Yes. I take ...",1,1,0,0,73.0,1,White,2021-05-17,541,1,0.585976
762,938868.0,523.0,523_2022-03-29,"Hello. Good. Yeah, he's good. Sure. Oh. Well. ...",2,0,1,1,72.0,1,White,2022-03-29,797,1,0.602682
763,938702.0,600.0,600_2024-07-17,ello? Yes. Who? Put you? Behave yourself. All ...,0,0,0,0,89.0,1,White,2024-07-17,1604,1,0.664875
764,938703.0,158.0,158_2023-09-21,"000000] Hello. Hello. So saying. Yes, I do. Oh...",0,0,0,0,77.0,1,Black or African American,2023-09-21,545,0,0.588911


In [95]:
print("Black propensity scores:")
print(f"  Min: {black['race_propensity_score'].min():.3f}")
print(f"  Max: {black['race_propensity_score'].max():.3f}")
print(f"  Mean: {black['race_propensity_score'].mean():.3f}")
print(f"  Count: {len(black)}")

print("\nWhite propensity scores:")
print(f"  Min: {white['race_propensity_score'].min():.3f}")
print(f"  Max: {white['race_propensity_score'].max():.3f}")
print(f"  Mean: {white['race_propensity_score'].mean():.3f}")
print(f"  Count: {len(white)}")

Black propensity scores:
  Min: 0.533
  Max: 0.918
  Mean: 0.680
  Count: 383

White propensity scores:
  Min: 0.451
  Max: 0.900
  Mean: 0.718
  Count: 924


## Subgroup Prediction Analysis: Testing Model Fairness Across Demographics

### What Are We Doing?

We're building **separate prediction models** for each demographic subgroup (Males, Females, Whites, Blacks) to test if our text-based prediction of CEL Total scores works equally well across different groups.

### Why Is This Important?

**Goal:** Ensure fairness and check for algorithmic bias
- If the model predicts well for Whites but poorly for Blacks → potential bias
- If the model predicts well for Males but poorly for Females → gender bias
- Equal performance across groups → fair and robust model

### The Process

1. **Separate Matched Data by Subgroups**
   - Males (397 participants)
   - Females (397 participants)
   - Whites (383 participants)
   - Blacks (383 participants)

2. **For Each Subgroup:**
   - Convert text transcripts to TF-IDF features
   - Split by participant ID (80% train, 20% test)
   - Train Random Forest model to predict CEL Total from text
   - Evaluate using Mean Absolute Error (MAE)

3. **Compare Performance**
   - Lower MAE = better predictions
   - Similar MAE across groups = fair model
   - Large differences = potential bias or data quality issues

### What We're Testing

**Research Question:** Can we predict emotional loneliness (CEL Total) from speech patterns equally well across:
- Gender groups?
- Racial groups?

### Expected Outcome

If MAE is similar across all groups (~1.3-1.6), it suggests:
- ✓ The model works fairly across demographics
- ✓ Text features capture loneliness patterns consistently
- ✓ No systematic bias in prediction capability

In [98]:
stratified_male_df = final_gender[final_gender['Gender']==0]    # ✓ Male = 0
stratified_female_df = final_gender[final_gender['Gender']==1]  # ✓ Female = 1
stratified_white_df = final_race[final_race['Race'].isin(white_race)]  # ✓ Race is still string
stratified_black_df = final_race[final_race['Race']=='Black or African American']  # ✓ Race is still string

# List of DataFrames
dataframes = {"Females":stratified_female_df, "Males":stratified_male_df, "Whites":stratified_white_df, "Blacks":stratified_black_df}

# Initialize a list to store results
mae_results = []

# Loop through each dataframe
for name, dataframe in dataframes.items():
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(dataframe['Text'])
    y = dataframe['CEL Total']
    
    # Get unique IDs and split them
    unique_ids = dataframe['New ID'].unique()

    random_state = np.random.RandomState(42)
    random_state.shuffle(unique_ids)

    train_size = int(len(unique_ids) * 0.8)
    train_ids = unique_ids[:train_size]
    test_ids = unique_ids[train_size:]

    train_mask = dataframe['New ID'].isin(train_ids)
    test_mask = dataframe['New ID'].isin(test_ids)

    train_mask = train_mask.to_numpy()
    test_mask = test_mask.to_numpy()

    X_train = X[train_mask]
    X_test = X[test_mask]

    training_dataset = dataframe[train_mask]
    testing_dataset = dataframe[test_mask]

    y_train = training_dataset['CEL Total']
    y_test = testing_dataset['CEL Total']

    print(f"Training set shape for {name}: {X_train.shape}")
    print(f"Test set shape for {name}: {X_test.shape}")

    # Model training
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Prediction and evaluation
    predicted_scores = model.predict(X_test)
    model_score = model.score(X_test,y_test)
    mae = ((y_test - predicted_scores).abs()).mean()

    print(f"{name} model score: {model_score}")
    print(f"{name} predictions Mean Absolute Error (MAE): {mae}")
    print("----------------------------x------------------------------------")
    mae_results.append(mae)

# You can now access the mae_results list, which contains the MAE for each dataframe
print("MAE results for all datasets:", mae_results)

Training set shape for Females: (317, 11935)
Test set shape for Females: (80, 11935)
Females model score: 0.05740081774253514
Females predictions Mean Absolute Error (MAE): 1.6806249999999998
----------------------------x------------------------------------
Training set shape for Males: (317, 12692)
Test set shape for Males: (80, 12692)
Males model score: 0.06991909246575356
Males predictions Mean Absolute Error (MAE): 1.7127500000000002
----------------------------x------------------------------------
Training set shape for Whites: (296, 12103)
Test set shape for Whites: (87, 12103)
Whites model score: 0.07093888524590153
Whites predictions Mean Absolute Error (MAE): 1.211954022988506
----------------------------x------------------------------------
Training set shape for Blacks: (311, 9163)
Test set shape for Blacks: (72, 9163)
Blacks model score: -0.022513573515092533
Blacks predictions Mean Absolute Error (MAE): 1.5711111111111107
----------------------------x----------------------

In [100]:
print(stratified_male_df.shape)
print(stratified_female_df.shape)
print(stratified_white_df.shape)
print(stratified_black_df.shape)

(397, 14)
(397, 14)
(383, 15)
(383, 15)


In [ ]:
#stratified_male_df = stratified_male_df[['Filename', 'Text', 'KlaatchID', 'Date', 'CEL Total', 'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'status']]
#stratified_female_df = stratified_female_df[['Filename', 'Text', 'KlaatchID', 'Date', 'CEL Total', 'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'status']]
#stratified_white_df = stratified_white_df[['Filename', 'Text', 'KlaatchID', 'Date', 'CEL Total', 'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'status']]
#stratified_black_df = stratified_black_df[['Filename', 'Text', 'KlaatchID', 'Date', 'CEL Total', 'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age', 'status']]

,New ID,Old ID,Filename,Text,CEL Total,CELVAL1,CELVAL2,CELVAL3,Age,Gender,Race,Date,word_count,gender_propensity_score
0,938906.0,643.0,643_2021-12-17,"Yes. From where? Okay, Diana. Oh, yeah, yeah...",4,1,1,2,79.0,0,Black or African American,2021-12-17,2208,0.021498
1,938727.0,490.0,490_2022-03-29,"Hello, Lisa. Has been. I've probably been. Rig...",3,1,1,1,84.0,1,Black or African American,2022-03-29,821,0.991453
3,938663.0,48.0,48_2022-05-27,"No. Hello, Lisa. Okay. I checked. Mine, too. O...",2,1,1,0,75.0,1,Black or African American,2022-05-27,677,0.989632
5,938733.0,606.0,606_2022-07-18,"Oh, please leave a message. Oh, lousy today. V...",1,0,1,0,94.0,1,Black or African American,2022-07-18,1660,0.993513
6,938731.0,766.0,766_2024-09-27,ot sauce? Yes. Okay. How are you? No. Yeah. Ye...,3,1,1,1,66.0,1,Black or African American,2024-09-27,634,0.98693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
753,938705.0,24.0,24_2021-02-09,"Yeah. Good. Okay. Yeah, I'll do it. Okay. I...",5,1,1,3,93.0,0,Black or African American,2021-02-09,282,0.029204
756,938682.0,545.0,545_2024-07-24,"ello? Yes. Speaker2: I'm fine. Lorna, how are...",4,2,1,1,65.0,1,Black or African American,2024-07-24,333,0.986356
758,938751.0,386.0,386_2024-08-07,"ello. Hi. What happened? Oh, okay. No. Okay. W...",3,1,1,1,80.0,1,Black or African American,2024-08-07,507,0.990605
760,938672.0,79.0,79_2022-02-01,"Fine, thank you. How are you? Yes, sure. It wa...",3,1,1,1,90.0,1,Black or African American,2022-02-01,297,0.992584


In [ ]:
def connect_to_db():
    return mysql.connector.connect(
        host='127.0.0.1',
        database='Audio_features',
        username="vinmayk", password="yA2Siru4l4yfEf1a"
    )
    
def create_table_if_not_exists(table_name):
    """
    Creates the table if it does not exist.
    """
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        message_id VARCHAR(191) NOT NULL,
        message TEXT,
        Date DATE,
        CEL_Total INT,
        CELVAL1 INT,
        CELVAL2 INT,
        CELVAL3 INT,
        klaatch_id INT,
        PRIMARY KEY (message_id)
    );
    """
    connection = connect_to_db()
    cursor = connection.cursor()
    try:
        cursor.execute(create_table_query)
        connection.commit()
        print(f"Table {table_name} is ready.")
    except mysql.connector.Error as err:
        print(f"Error creating table {table_name}: {err}")
    finally:
        cursor.close()
        connection.close()


def insert_data(df, table_name):
    """
    Inserts data into the specified table.
    Args:
    df (pd.DataFrame): The DataFrame containing the data to insert.
    table_name (str): The name of the database table to insert data into.
    """
    
    create_table_if_not_exists(table_name)
    
    # Establish a connection to the database
    connection = connect_to_db()
    cursor = connection.cursor()

    # Construct the INSERT query dynamically based on the table name
    insert_query = f"""
    INSERT IGNORE INTO {table_name} (
        message_id, message, Date, CEL_Total, CELVAL1, CELVAL2, CELVAL3, klaatch_id
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    # Convert DataFrame to a list of tuples, including all required columns
    data = df[['Filename', 'Text', 'Date', 'CEL Total',
               'CELVAL1', 'CELVAL2', 'CELVAL3', 'Old ID']].values.tolist()
    
    # Insert data
    try:
        cursor.executemany(insert_query, data)
        connection.commit()
        print(f"Inserted {cursor.rowcount} rows successfully into table {table_name}.")
    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        cursor.close()
        connection.close()

# Example usage
insert_data(stratified_male_df, "stratified_male")
insert_data(stratified_female_df, "stratified_female")
insert_data(stratified_black_df, "stratified_black")
insert_data(stratified_white_df, "stratified_white")


Table stratified_male is ready.
Inserted 397 rows successfully into table stratified_male.
Table stratified_female is ready.
Inserted 397 rows successfully into table stratified_female.
Table stratified_black is ready.
Inserted 383 rows successfully into table stratified_black.
Table stratified_white is ready.
Inserted 383 rows successfully into table stratified_white.


In [19]:
print(len(stratified_male_df))
print(len(stratified_female_df))
print(len(stratified_white_df))
print(len(stratified_black_df))

397
397
383
383


## DLATK

### DLATK Feature Extraction with Librosa Features

Running DLATK predictions using Librosa acoustic features (MFCCs, Chroma, Spectral Contrast, Tonnetz)

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:46:06
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 38)
[COMBINED FEATS:

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:46:43
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 38)
[COMBINED FEATS:

In [3]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:50:22
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CELVAL1', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 38)
[COMBINED FEATS:

In [4]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:50:36
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL2', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 38)
[COMBINED FEATS:

### DLATK Feature Extraction with OpenSmile Features

Running DLATK predictions using OpenSmile eGeMAPSv02 acoustic features (88 features)

In [5]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:56:12
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 88)
[COMBINED FEATS:

In [6]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:56:32
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 88)
[COMBINED FEATS:

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:56:56
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS:

In [8]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:57:13
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL1', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS:

In [9]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:57:49
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL2', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [10]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 17:59:08
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [11]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:00:41
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CEL_Total', 'CELVAL1', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [12]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:03:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CEL_Total', 'CELVAL2', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [13]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:17:45
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [14]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:19:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [15]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:22:02
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

### DLATK Feature Extraction with Whisper Features

Running DLATK predictions using Whisper speech embeddings (mean and median aggregations)

In [16]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:23:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [17]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:24:08
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 3

In [18]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:25:42
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 3

In [19]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:26:51
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 3

In [20]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:27:57
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL3', 'CELVAL1', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 3

In [22]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 03:51:37
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEAT

In [23]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 03:51:54
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEAT

In [24]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 03:52:09
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 103)
[COMBINED FEAT

In [25]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 03:52:24
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CEL_Total', 'CELVAL2', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 103)
[COMBINED FEAT

### DLATK Feature Extraction with Text Features

Running DLATK predictions using text-based features (categories, LIWC, word embeddings)

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:54:36
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 51)
[COMBINED FEATS:

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:57:42
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 51)
[COMBINED FEATS:

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:58:08
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 51)
[COMBINED FEATS:

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 18:58:26
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL2', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 51)
[COMBINED FEATS:

In [31]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 19:08:16
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL2', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1486)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 1486)
[COMBINED FE

In [32]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 19:09:31
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CELVAL1', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1290)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 1290)
[COMBINED FE

In [33]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 19:10:35
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL1', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 1484)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 1484)
[COMBINED FE

In [35]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 19:13:09
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CELVAL1', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 1204)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 1204)
[COMBINED FE

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:00:02
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: Sta

In [27]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SFemale50_cp_w$stratified_female$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:01:03
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CELVAL1', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: Sta

In [28]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Swhite50_cp_w$stratified_white$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:02:01
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: Sta

In [29]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Sblack50_cp_w$stratified_black$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:02:52
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL2', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: Sta

In [30]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SMale50_cp_w$stratified_male$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:03:42
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
   (feature group: 4): [Initial size: 377]
   (feature group: 5):

#### SVR Predictions with OpenSmile Features

Support Vector Regression model predictions using OpenSmile acoustic features

In [31]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SFemale50_cp_w$stratified_female$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:05:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL2', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
   (feature group: 4): [Initial size: 377]
   (feature group: 5):

In [32]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Swhite50_cp_w$stratified_white$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:06:18
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
   (feature group: 4): [Initial size: 356]
   (feature group: 5):

In [33]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Sblack50_cp_w$stratified_black$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model ridgehighcv --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:07:27
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
   (feature group: 4): [Initial size: 356]
   (feature group: 5):

### SVR model

In [48]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:16:19
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 38)
[COMBINED FEATS:

In [49]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:16:28
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 38)
[COMBINED FEATS: P

In [50]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:16:38
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL1', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 38)
[COMBINED FEATS:

In [51]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:16:48
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 38)
[COMBINED FEATS:

In [52]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:16:57
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 88)
[COMBINED FEATS: P

#### SVR Predictions with Whisper Features

Support Vector Regression using Whisper speech embeddings (mean and median)

In [53]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:17:08
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 88)
[COMBINED FEATS: P

In [54]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:17:20
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS: P

In [55]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:17:31
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS:

In [56]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:17:43
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [57]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:18:07
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

In [58]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:18:32
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL2', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEATS:

In [59]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:18:56
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [60]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:19:20
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEATS:

In [61]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:19:45
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEAT

#### SVR Predictions with Demographic Subgroups

Support Vector Regression predictions stratified by gender and demographic groups

In [62]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:20:10
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [63]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:20:33
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 512)
[COMBINED FEAT

In [64]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:20:56
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)

In [65]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:21:37
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL1', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 3

In [66]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:22:19
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL3', 'CELVAL1', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 3

In [67]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:23:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL2', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 3

In [34]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:24:08
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEAT

In [35]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:24:20
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEAT

In [36]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:24:31
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL1', 'CELVAL3', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 103)
[COMBINED FEAT

#### SVR Predictions with Text Features

Support Vector Regression using text-based features (LIWC, categories, word embeddings)

In [37]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 04:24:41
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CELVAL3', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 103)
[COMBINED FEAT

In [72]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_SMale50_cp_w$stratified_male$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:24:28
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL3', 'CELVAL1', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 51)
[COMBINED FEATS: P

In [73]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_SFemale50_cp_w$stratified_female$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:24:39
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 51)
[COMBINED FEATS:

In [74]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_Swhite50_cp_w$stratified_white$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:24:50
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL1', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 51)
[COMBINED FEATS: P

In [75]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_Sblack50_cp_w$stratified_black$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:25:01
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CEL_Total', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 51)
[COMBINED FEATS: P

In [76]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:25:12
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL1', 'CEL_Total', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1486)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 1486)
[COMBINED FEAT

In [77]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:25:48
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1290)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (301, 1290)
[COMBINED FEAT

In [78]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:26:20
-----
Loading Outcomes and Getting Groups for: {'CELVAL1', 'CEL_Total', 'CELVAL2', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 1484)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 1484)
[COMBINED FEAT

In [79]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-06 20:26:52
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 1204)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (285, 1204)
[COMBINED FE

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SMale50_cp_w$stratified_male$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:04:06
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CELVAL1', 'CELVAL2', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: Sta

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SFemale50_cp_w$stratified_female$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:04:45
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: Sta

#### ExtraTrees Predictions with OpenSmile Features

Extra Trees Regressor model predictions using OpenSmile acoustic features

In [3]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Swhite50_cp_w$stratified_white$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:05:21
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL3', 'CELVAL1', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: Sta

In [4]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Sblack50_cp_w$stratified_black$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:05:57
-----
Loading Outcomes and Getting Groups for: {'CELVAL3', 'CEL_Total', 'CELVAL1', 'CELVAL2'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: Sta

In [5]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SMale50_cp_w$stratified_male$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:06:28
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CEL_Total', 'CELVAL3', 'CELVAL1'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
   (feature group: 4): [Initial size: 377]
   (feature group: 5):

In [6]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_SFemale50_cp_w$stratified_female$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:07:46
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CELVAL3', 'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 377 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 377]
   (feature group: 1): [Initial size: 377]
   (feature group: 2): [Initial size: 377]
   (feature group: 3): [Initial size: 377]
   (feature group: 4): [Initial size: 377]
   (feature group: 5):

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Swhite50_cp_w$stratified_white$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:09:00
-----
Loading Outcomes and Getting Groups for: {'CELVAL2', 'CELVAL1', 'CEL_Total', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
   (feature group: 4): [Initial size: 356]
   (feature group: 5):

In [8]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_Sblack50_cp_w$stratified_black$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total CELVAL1 CELVAL2 CELVAL3 \
    --nfold_test_regression --model svr --fold_column fold



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-13 12:10:09
-----
Loading Outcomes and Getting Groups for: {'CEL_Total', 'CELVAL2', 'CELVAL1', 'CELVAL3'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 356 (5 Folds)]


|COMBO: ()|

= CELVAL1 (w/ lang.)=
---------------------
Fold 0 
   (feature group: 0): [Initial size: 356]
   (feature group: 1): [Initial size: 356]
   (feature group: 2): [Initial size: 356]
   (feature group: 3): [Initial size: 356]
   (feature group: 4): [Initial size: 356]
   (feature group: 5):

### Extratrees model

In [16]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 17:39:03
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (303, 38)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 05:34:05
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 38)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 10:56:58
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 38)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

#### ExtraTrees Predictions with Whisper Features

Extra Trees Regressor using Whisper speech embeddings

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 10:57:58
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 38)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [3]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 10:58:52
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 88)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 88)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [4]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:00:24
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 88)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 88)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [5]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:01:40
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [6]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:03:12
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 88)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 88)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=Non

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:06:11
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [8]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:07:30
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [9]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:09:45
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [10]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold 



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:11:40
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [11]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:12:58
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [12]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:15:08
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [13]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:16:27
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [14]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:20:32
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 512)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 512)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [15]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:22:31
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
   (feature group: 1): [Initial size: 391]
   (feature group: 2): [Initial size: 391]
   (feature group: 3): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (301, 88)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True,

#### ExtraTrees Predictions with Demographic Subgroups

Extra Trees Regressor predictions stratified by demographic groups

In [16]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:23:24
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
   (feature group: 1): [Initial size: 400]
   (feature group: 2): [Initial size: 400]
   (feature group: 3): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (301, 88)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True,

In [17]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:24:15
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
   (feature group: 1): [Initial size: 374]
   (feature group: 2): [Initial size: 374]
   (feature group: 3): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (284, 88)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True,

In [18]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 11:25:05
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
   (feature group: 1): [Initial size: 373]
   (feature group: 2): [Initial size: 373]
   (feature group: 3): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 38)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (284, 88)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True,

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 13:48:59
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 13:49:23
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 103)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [3]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 13:52:01
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 374 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 374]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 103)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [4]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 13:52:16
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 103)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:04:07
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 101)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (303, 101)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [2]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:04:22
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 101)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (303, 101)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [3]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:04:38
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 298    Test size: 75]
 X[0]: (N, features): (298, 101)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (298, 101)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:04:53
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 298    Test size: 75]
 X[0]: (N, features): (298, 101)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (298, 101)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes=N

In [6]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 14:07:18
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 391 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 391]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1313)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 1313)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 14:07:59
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 400 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 400]
 [Train size: 301    Test size: 76]
 X[0]: (N, features): (301, 1313)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (301, 1313)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes

In [5]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:06:54
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 289    Test size: 71]
 X[0]: (N, features): (289, 1313)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (289, 1313)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes

#### ExtraTrees Predictions with Text Features

Extra Trees Regressor using text-based features and linguistic patterns

In [9]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 14:10:12
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 1313)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FEATS (from multiX)] Combined size: (284, 1313)
[COMBINED FEATS: Training regression model: extratrees]
model: ExtraTreesRegressor(bootstrap=False, criterion='mse', max_depth=None,
          max_features='auto', max_leaf_nodes

In [6]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:08:33
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
   (feature group: 1): [Initial size: 379]
   (feature group: 2): [Initial size: 379]
 [Train size: 291    Test size: 74]
 X[0]: (N, features): (291, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (291, 101)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[2]: (N, features): (2

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:09:21
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
   (feature group: 1): [Initial size: 379]
   (feature group: 2): [Initial size: 379]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (285, 101)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[2]: (N, features): (2

In [8]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:10:02
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
   (feature group: 1): [Initial size: 373]
   (feature group: 2): [Initial size: 373]
 [Train size: 285    Test size: 70]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (285, 101)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[2]: (N, features): (2

In [9]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:10:52
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
   (feature group: 1): [Initial size: 373]
   (feature group: 2): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (284, 101)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[2]: (N, features): (2

In [10]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_male$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:11:36
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
   (feature group: 1): [Initial size: 379]
   (feature group: 2): [Initial size: 379]
   (feature group: 3): [Initial size: 379]
   (feature group: 4): [Initial size: 379]
   (feature group: 5): [Initial size: 379]
   (feature group: 6): [Initial size: 379]
 [Train size: 291    Test size: 74]
 X[0]: (N, features): (291, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=T

In [11]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_female$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:13:03
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
   (feature group: 1): [Initial size: 379]
   (feature group: 2): [Initial size: 379]
   (feature group: 3): [Initial size: 379]
   (feature group: 4): [Initial size: 379]
   (feature group: 5): [Initial size: 379]
   (feature group: 6): [Initial size: 379]
 [Train size: 285    Test size: 71]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=T

In [12]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_white$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:14:20
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
   (feature group: 1): [Initial size: 373]
   (feature group: 2): [Initial size: 373]
   (feature group: 3): [Initial size: 373]
   (feature group: 4): [Initial size: 373]
   (feature group: 5): [Initial size: 373]
   (feature group: 6): [Initial size: 373]
 [Train size: 285    Test size: 70]
 X[0]: (N, features): (285, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=T

In [13]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$stratified_black$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:15:43
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 373 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 373]
   (feature group: 1): [Initial size: 373]
   (feature group: 2): [Initial size: 373]
   (feature group: 3): [Initial size: 373]
   (feature group: 4): [Initial size: 373]
   (feature group: 5): [Initial size: 373]
   (feature group: 6): [Initial size: 373]
 [Train size: 284    Test size: 72]
 X[0]: (N, features): (284, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=T

In [14]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:17:02
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1468 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1468]
   (feature group: 1): [Initial size: 1468]
   (feature group: 2): [Initial size: 1468]
   (feature group: 3): [Initial size: 1468]
   (feature group: 4): [Initial size: 1468]
   (feature group: 5): [Initial size: 1468]
   (feature group: 6): [Initial size: 1468]
 [Train size: 1082    Test size: 272]
 X[0]: (N, features): (1082, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, 

In [15]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' 'feat$cat_klaatch_senten2_lda_cp_w$merged_data$message_id$1gra' 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold



/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-11 15:22:54
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1468 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1468]
   (feature group: 1): [Initial size: 1468]
   (feature group: 2): [Initial size: 1468]
 [Train size: 1082    Test size: 272]
 X[0]: (N, features): (1082, 103)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[1]: (N, features): (1082, 101)
  [Applying StandardScaler to X[1]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[2]: (N, featu

In [10]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-25 16:21:07
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]
^C


In [59]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$librosa_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:26:26
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 38)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 38)
[COMBINED FEATS: Training regression mode

## XGBoost Model Predictions

Running XGBoost gradient boosting predictions across different feature types and subgroups

In [60]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$opensmile_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:26:41
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 88)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 88)
[COMBINED FEATS: Training regression mo

In [61]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_mean_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:27:03
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 512)
[COMBINED FEATS: Training regression 

In [62]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$whisper_median_n$merged_data$message_id' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:28:28
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 512)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 512)
[COMBINED FEATS: Training regression 

In [ ]:
 'feat$opensmile_n$merged_data$message_id' 'feat$whisper_mean_n$merged_data$message_id' 'feat$whisper_median_n$merged_data$message_id' 

In [55]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_LIWC2015_lw$merged_data$message_id$1gra' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:21:07
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 74)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 74)
[COMBINED FEATS: Training regression mode

In [56]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_Klaatch_50_cp_w$merged_data$message_id$1gra' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:21:28
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1083    Test size: 270]
 X[0]: (N, features): (1083, 51)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1083, 51)
[COMBINED FEATS: Training regression mo

In [57]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$1to3gram$merged_data$message_id$0_05$pmi6_0' \
    --outcome_table merged_data  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --csv --output overall_csv



/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 03:21:46
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 1354 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 1354]
 [Train size: 1084    Test size: 270]
 X[0]: (N, features): (1084, 1313)
  [Applying StandardScaler to X[0]: StandardScaler()]
[COMBINED FEATS (from multiX)] Combined size: (1084, 1313)
[COMBINED FEATS: Training regressio

In [1]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -g message_id \
-f 'feat$1to3gram$stratified_male$message_id' \
    --estimate_lda_topics \
    --mallet_path /home/karthik9/mallet-2.0.8RC3/bin/mallet \
    --lda_lexicon_name SMale_LDA_50 \
    --num_lda_threads 10 \
    --save_lda_files SMale_LDA_50 \
    --num_topics 50

/home/vinmayk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
TopicExtractor: gensim Mallet wrapper unavailable, using Mallet directly.

-----
DLATK Interface Initiated: 2025-02-11 16:00:45
-----
Writing joined feature lines file: SMale_LDA_50/lda_msgs.txt.
Yielding values with zeros for 1354 groups * 432724 feats.
Wrote joined features file to: SMale_LDA_50/lda_msgs.txt
Estimating LDA topics. This may take a long time!
Estimating LDA topics using Mallet.
Traceback (most recent call last):
  File "/home/vinmayk/dlatk/dlatkInterface.py", line 2266, in <module>
    main()
  File "/home/vinmayk/dlatk/dlatkInterface.py", line 1162, in main
    state_file = lda_estimator.estimate_to

## Binary Classification: Loneliness Detection

Predicting loneliness as a binary outcome (lonely = 0, not lonely = 1)

## XGB

In [8]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_data$moe_male$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_data$moe_female$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_data$moe_white$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_data$moe_black$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-26 16:49:45
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 6)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FE

In [7]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_text$moe_male$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_text$moe_female$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_text$moe_white$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_text$moe_black$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-26 16:48:29
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 3)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FE

In [9]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_audio$moe_male$message_id' \
    --outcome_table stratified_male  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_audio$moe_female$message_id' \
    --outcome_table stratified_female  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_audio$moe_white$message_id' \
    --outcome_table stratified_white  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 


!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$combined_audio$moe_black$message_id' \
    --outcome_table stratified_black  --group_freq_thresh 1 \
    --outcomes CEL_Total \
    --nfold_test_regression --model extratrees --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-03-26 16:53:34
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
    ***explicit fold labels specified, not splitting again***
[number of groups: 379 (5 Folds)]


|COMBO: ()|

= CEL_Total (w/ lang.)=
-----------------------
Fold 0 
   (feature group: 0): [Initial size: 379]
 [Train size: 303    Test size: 76]
 X[0]: (N, features): (303, 3)
  [Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
[COMBINED FE

## Topic Analysis and Interpretation

Retrieving and analyzing top topics/words associated with predictions using MySQL database queries

In [1]:
import sys
print(sys.executable)

/data/anaconda2/envs/dlatk/bin/python


In [1]:
!python /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t la_table -g message_id --add_ngrams -n 1 2 3 --combine_feat_tables 1to3gram



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-08 14:05:39
-----
query: SELECT column_type FROM information_schema.columns WHERE table_schema='Audio_features' AND table_name='la_table' AND column_name='message_id'
SQL QUERY: DROP TABLE IF EXISTS feat$1gram$la_table$message_id
SQL QUERY: CREATE TABLE feat$1gram$la_table$message_id ( id BIGINT(16) UNSIGNED NOT NULL AUTO_INCREMENT PRIMARY KEY, group_id varchar(191), feat VARCHAR(36) CHARACTER SET utf8mb4 COLLATE utf8mb4_bin, value INTEGER, group_norm 

In [2]:
!python /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t la_table -c message_id --add_lex_table -l LIWC2022 --liwc_normalization



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-08 14:09:12
-----
###################################################################
  specify --weighted_lexicon so the weights won't be used
###################################################################
query: SELECT column_type FROM information_schema.columns WHERE table_schema='Audio_features' AND table_name='la_table' AND column_name='message_id'
SQL QUERY: DROP TABLE IF EXISTS feat$cat_LIWC2022_lw$la_table$message_id$1gra
SQL QUERY: CREATE

## lonely = 0 & not lonely = 1

In [14]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t la_table -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table la_table  --group_freq_thresh 1 \
    --outcomes status \
    --nfold_test_classifiers --model etc --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-08 15:49:01
-----
Loading Outcomes and Getting Groups for: {'status'}
    ***explicit fold labels specified, not splitting or stratifying again***


|COMBO: ()|

= status (w/ lang.)=
--------------------

Fold 0 
   (feature group: 0): [Initial size: 1468]
[Train size: 1174    Test size: 294]
[Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[0]: (N, features): (1174, 103)
[Training classification model: etc]

In [ ]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t la_table -c message_id \
-f 'feat$cat_LIWC2022_lw$merged_data$message_id$1gra' \
    --outcome_table la_table  --group_freq_thresh 1 \
    --outcomes status \
    --nfold_test_classifiers --model etc --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-08 14:33:32
-----
Loading Outcomes and Getting Groups for: {'status'}
    ***explicit fold labels specified, not splitting or stratifying again***


|COMBO: ()|

= status (w/ lang.)=
--------------------

Fold 0 
   (feature group: 0): [Initial size: 1468]
[Train size: 1174    Test size: 294]
[Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[0]: (N, features): (1174, 3)
[Training classification model: etc]
m

In [12]:
!python3 /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t la_table -c message_id \
-f 'feat$combined$merged_data$message_id' \
    --outcome_table la_table  --group_freq_thresh 1 \
    --outcomes status \
    --nfold_test_classifiers --model etc --fold_column fold --turn_off_backoff_model 



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-08 14:33:47
-----
Loading Outcomes and Getting Groups for: {'status'}
    ***explicit fold labels specified, not splitting or stratifying again***


|COMBO: ()|

= status (w/ lang.)=
--------------------

Fold 0 
   (feature group: 0): [Initial size: 1468]
[Train size: 1174    Test size: 294]
[Applying StandardScaler to X[0]: StandardScaler(copy=True, with_mean=True, with_std=True)]
 X[0]: (N, features): (1174, 6)
[Training classification model: etc]
m

In [1]:
!python /home/vinmayk/dlatk/dlatkInterface.py -d Audio_features -t merged_data -c message_id \
-f 'feat$cat_Final_Klaatch_100_cp_w$merged_data$message_id$1gra' \
  --outcome_table merged_data --group_freq_thresh 5 \
  --outcomes CEL_Total --output_name Final_Klaatch_100 \
  --topic_tagcloud --make_topic_wordcloud --topic_lexicon Final_Klaatch_100_freq_t50ll \
  --tagcloud_colorscheme bluered --csv



/home/vinmayk/.local/lib/python3.5/site-packages/xgboost/__init__.py:29: FutureWarning: Python 3.5 support is deprecated; XGBoost will require Python 3.6+ in the near future. Consider upgrading to Python 3.6+.
  FutureWarning)
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/statsmodels/compat/pandas.py:56: FutureWarning: The pandas.core.datetools module is deprecated and will be removed in a future version. Please use the pandas.tseries module instead.
  from pandas.core import datetools

-----
DLATK Interface Initiated: 2025-04-22 18:11:07
-----
Loading Outcomes and Getting Groups for: {'CEL_Total'}
Yielding data over ['CEL_Total'], adjusting for: [].
Yielding norms with zeros (1465 groups * 101 feats).
/data/anaconda2/envs/dlatk/lib/python3.5/site-packages/scipy/stats/stats.py:3021: RuntimeWarning: invalid value encountered in double_scalars
  r = r_num / r_den
Generating Correlation Matrix.
 print to file: Final_Klaatch_100.csv
Namespace(IDP=False, adaptationfactors=None, a

import mysql.connector
import pandas as pd
# Connect to MySQL database
conn = mysql.connector.connect(
    host='127.0.0.1',
    database='dlatk_lexica',
    username='vinmayk', password='yA2Siru4l4yfEf1a'
)
# Create a cursor
cursor = conn.cursor()
# Initialize variables separately
cursor.execute("SET @num := 0;")
cursor.execute("SET @type := '';")
# Define the query
query = """
SELECT category,
       GROUP_CONCAT(term ORDER BY weight DESC SEPARATOR ', ') AS termy
FROM (
    SELECT category, term, weight,
           (@num := IF(@type = category, @num + 1, 1)) AS row_number,
           (@type := category) AS dummy
    FROM `Final_Klaatch_100_freq_t50ll`
    ORDER BY category, weight DESC
) AS x
WHERE x.row_number <= 10
GROUP BY category;
"""
# Fetch data into a pandas DataFrame
new_df = pd.read_sql_query(query, con=conn)
# Close cursor and connection
cursor.close()
conn.close()
# Print the DataFrame
new_df.to_csv('Final_100topics_Klaatch.csv', index=False)

In [2]:
import pandas as pd
import json

import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text

db = sqlalchemy.engine.url.URL(drivername='mysql+pymysql',
                            host='127.0.0.1',
                            database='Audio_features',
                            username = "vinmayk", password = "yA2Siru4l4yfEf1a", port=None,
                            query={'read_default_file': '~/.my.cnf', 'charset':'utf8mb4'})
engine = sqlalchemy.create_engine(db)

In [3]:
table = 'feat$cat_Final_Klaatch_50_cp_w$merged_data$message_id$1gra'
msg_table = 'merged_data'

In [4]:
# read tables
tbl = pd.read_sql_table(table, engine)
msg_tbl = pd.read_sql_table(msg_table, engine)
#msg_tbl = msg_tbl[['message_id', 'message', 'KlaatchID_x', 'Date_x' , 'CEL_Total', 'CELVAL1', 'CELVAL2', 'CELVAL3']]

In [5]:
tbl

,id,group_id,feat,value,group_norm
0,1,103_2021-04-21,12,93,0.195711
1,2,103_2021-04-21,31,28,0.009537
2,3,103_2021-04-21,83,8,0.000445
3,4,103_2021-04-21,10,30,0.006536
4,5,103_2021-04-21,18,51,0.023640
...,...,...,...,...,...
147685,147686,997606_2024-10-25,15,168,0.007278
147686,147687,997606_2024-10-25,28,46,0.001827
147687,147688,997606_2024-10-25,74,40,0.000766
147688,147689,997606_2024-10-25,76,1287,0.241210


In [6]:
# drop row where feat == '_intercept'
tbl = tbl[tbl['feat'] != '_intercept']
# convert feat to int
tbl['feat'] = tbl['feat'].astype(int)

tbl = tbl.rename(columns={'feat': 'category'})


/tmp/ipykernel_1384335/3160203219.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl['feat'] = tbl['feat'].astype(int)


In [7]:
tbl = tbl.merge(msg_tbl, left_on='group_id', right_on='message_id', how='left')

In [8]:
tbl

,id,group_id,category,value,group_norm,message_id,message,Date,CEL_Total,CELVAL1,CELVAL2,CELVAL3,klaatch_id,fold
0,1,103_2021-04-21,12,93,0.195711,103_2021-04-21,Hello. What's going on today? It's out in I lo...,2021-04-21,1,0,0,1,103,3
1,2,103_2021-04-21,31,28,0.009537,103_2021-04-21,Hello. What's going on today? It's out in I lo...,2021-04-21,1,0,0,1,103,3
2,3,103_2021-04-21,83,8,0.000445,103_2021-04-21,Hello. What's going on today? It's out in I lo...,2021-04-21,1,0,0,1,103,3
3,4,103_2021-04-21,10,30,0.006536,103_2021-04-21,Hello. What's going on today? It's out in I lo...,2021-04-21,1,0,0,1,103,3
4,5,103_2021-04-21,18,51,0.023640,103_2021-04-21,Hello. What's going on today? It's out in I lo...,2021-04-21,1,0,0,1,103,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146220,147685,997606_2024-10-25,99,28,0.001594,997606_2024-10-25,"Hello. Okay. Nope. Sure. Yes. Yep. Okay. Well,...",2024-10-25,9,2,3,4,997606,4
146221,147686,997606_2024-10-25,15,168,0.007278,997606_2024-10-25,"Hello. Okay. Nope. Sure. Yes. Yep. Okay. Well,...",2024-10-25,9,2,3,4,997606,4
146222,147687,997606_2024-10-25,28,46,0.001827,997606_2024-10-25,"Hello. Okay. Nope. Sure. Yes. Yep. Okay. Well,...",2024-10-25,9,2,3,4,997606,4
146223,147688,997606_2024-10-25,74,40,0.000766,997606_2024-10-25,"Hello. Okay. Nope. Sure. Yes. Yep. Okay. Well,...",2024-10-25,9,2,3,4,997606,4


In [9]:
# subset tbl to where message > 5 words
tbl_5_words = tbl[tbl['message'].str.split().str.len() > 5]

In [10]:
top_10_msgs = tbl.groupby('category').apply(lambda x: x.nlargest(10, 'group_norm')).reset_index(drop=True)


/tmp/ipykernel_1384335/676043890.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_10_msgs = tbl.groupby('category').apply(lambda x: x.nlargest(10, 'group_norm')).reset_index(drop=True)


In [11]:
top_10_msgs

,id,group_id,category,value,group_norm,message_id,message,Date,CEL_Total,CELVAL1,CELVAL2,CELVAL3,klaatch_id,fold
0,3475,128_2021-04-28,0,15,0.013553,128_2021-04-28,"Hello. Okay, surprise. Okay. I took care of th...",2021-04-28,1,1,0,0,128,3
1,132522,777_2023-07-31,0,131,0.013215,777_2023-07-31,"Oh, hello. Yeah, Hi. Good. I didn't. You cut o...",2023-07-31,4,1,2,1,777,4
2,49053,507_2021-02-09,0,8,0.013131,507_2021-02-09,So. Oh. Sure. People could. Yes. So I'd watch ...,2021-02-09,5,1,1,3,507,0
3,99723,679_2022-09-08,0,280,0.011227,679_2022-09-08,Hello. Hello. Yes. I'm out of breath right now...,2022-09-08,3,1,1,1,679,1
4,99622,679_2022-05-10,0,128,0.010738,679_2022-05-10,Hello. Yes. Mm hmm. All right. Okay. Would you...,2022-05-10,5,1,3,1,679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,98763,678_2022-08-18,99,18,0.011893,678_2022-08-18,"Hello. I'm busy, man, playing video games. Som...",2022-08-18,3,1,1,1,678,1
996,92305,661_2021-06-23,99,16,0.011484,661_2021-06-23,Go a new. Yeah. Good. That package to get ba...,2021-06-23,12,4,4,4,661,0
997,2110,117_2021-02-08,99,20,0.011465,117_2021-02-08,Hello. OK. Ok. Yesterday I went to KIPP's with...,2021-02-08,0,0,0,0,117,3
998,117986,734_2024-03-28,99,20,0.011382,734_2024-03-28,"h. Yes. I'm fine, thank you. Yes. I'm feeling ...",2024-03-28,3,1,1,1,734,0


In [12]:
top_10_msgs.columns

Index(['id', 'group_id', 'category', 'value', 'group_norm', 'message_id',
       'message', 'Date', 'CEL_Total', 'CELVAL1', 'CELVAL2', 'CELVAL3',
       'klaatch_id', 'fold'],
      dtype='object')

In [13]:
top_10_topics = pd.read_csv('/home/vinmayk/Final_100topics_Klaatch.csv')

In [14]:
top_10_msgs

,id,group_id,category,value,group_norm,message_id,message,Date,CEL_Total,CELVAL1,CELVAL2,CELVAL3,klaatch_id,fold
0,3475,128_2021-04-28,0,15,0.013553,128_2021-04-28,"Hello. Okay, surprise. Okay. I took care of th...",2021-04-28,1,1,0,0,128,3
1,132522,777_2023-07-31,0,131,0.013215,777_2023-07-31,"Oh, hello. Yeah, Hi. Good. I didn't. You cut o...",2023-07-31,4,1,2,1,777,4
2,49053,507_2021-02-09,0,8,0.013131,507_2021-02-09,So. Oh. Sure. People could. Yes. So I'd watch ...,2021-02-09,5,1,1,3,507,0
3,99723,679_2022-09-08,0,280,0.011227,679_2022-09-08,Hello. Hello. Yes. I'm out of breath right now...,2022-09-08,3,1,1,1,679,1
4,99622,679_2022-05-10,0,128,0.010738,679_2022-05-10,Hello. Yes. Mm hmm. All right. Okay. Would you...,2022-05-10,5,1,3,1,679,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,98763,678_2022-08-18,99,18,0.011893,678_2022-08-18,"Hello. I'm busy, man, playing video games. Som...",2022-08-18,3,1,1,1,678,1
996,92305,661_2021-06-23,99,16,0.011484,661_2021-06-23,Go a new. Yeah. Good. That package to get ba...,2021-06-23,12,4,4,4,661,0
997,2110,117_2021-02-08,99,20,0.011465,117_2021-02-08,Hello. OK. Ok. Yesterday I went to KIPP's with...,2021-02-08,0,0,0,0,117,3
998,117986,734_2024-03-28,99,20,0.011382,734_2024-03-28,"h. Yes. I'm fine, thank you. Yes. I'm feeling ...",2024-03-28,3,1,1,1,734,0


In [15]:
# merge top 3 msgs with top 10 words
top_10_msgs = top_10_msgs.merge(top_10_topics, on='category', how='left')
#print(top_10_msgs.columns)
#K=top_10_msgs[['id_x', 'id_y', 'group_id', 'category', 'message_id', 'message', 'termy']]
K=top_10_msgs[['id', 'group_id', 'category', 'message_id', 'message', 'termy']]
K
#top_10_msgs.columns


,id,group_id,category,message_id,message,termy
0,3475,128_2021-04-28,0,128_2021-04-28,"Hello. Okay, surprise. Okay. I took care of th...","dog, course, of course, course ,, of course ,,..."
1,132522,777_2023-07-31,0,777_2023-07-31,"Oh, hello. Yeah, Hi. Good. I didn't. You cut o...","dog, course, of course, course ,, of course ,,..."
2,49053,507_2021-02-09,0,507_2021-02-09,So. Oh. Sure. People could. Yes. So I'd watch ...,"dog, course, of course, course ,, of course ,,..."
3,99723,679_2022-09-08,0,679_2022-09-08,Hello. Hello. Yes. I'm out of breath right now...,"dog, course, of course, course ,, of course ,,..."
4,99622,679_2022-05-10,0,679_2022-05-10,Hello. Yes. Mm hmm. All right. Okay. Would you...,"dog, course, of course, course ,, of course ,,..."
...,...,...,...,...,...,...
995,98763,678_2022-08-18,99,678_2022-08-18,"Hello. I'm busy, man, playing video games. Som...","ow, jason, ow , tub, ow, ow ,, ow , ow, ambul..."
996,92305,661_2021-06-23,99,661_2021-06-23,Go a new. Yeah. Good. That package to get ba...,"ow, jason, ow , tub, ow, ow ,, ow , ow, ambul..."
997,2110,117_2021-02-08,99,117_2021-02-08,Hello. OK. Ok. Yesterday I went to KIPP's with...,"ow, jason, ow , tub, ow, ow ,, ow , ow, ambul..."
998,117986,734_2024-03-28,99,734_2024-03-28,"h. Yes. I'm fine, thank you. Yes. I'm feeling ...","ow, jason, ow , tub, ow, ow ,, ow , ow, ambul..."


In [ ]:
K[K['category']==18]['message'].to_list()

['TIL caregiver PTSD is real. HAHAHAHAHAHAHAHAHAJCDFHCCJNNVKHHJJJJN FUCK. \n\nThe human brain is an absolute travesty. Kill me.',
 "This is so hard It really really sucks and is very hard. That's about where I'm at right now. \n\nStay strong everyone.",
 'Conflicted CONFLICTED. Easier. Harder. Grieving. Distancing. Loving. Apathetic. This part of the dementia journey is a study in contradictions.\n\n[Curtain Up!](https://www.stumpedtowndementia.com/post/curtain-up)',
 "exelon I'm going to start.yhis soon, the patch. Does.smyone.hsvr any.expetirnces.yhey can share? thanks",
 "End-stage Alzheimer's and COVID My Dad, not even 70, is at the very end of a horrible decade with early-onset.  Today he tested positive for COVID, as did my Mom (no dementia).  Yup.",
 'Conflicted Easier. Harder. Grieving. Distancing. Loving. Apathetic. This part of the dementia journey is a study in contradictions.\n\n[Curtain Up!](https://www.stumpedtowndementia.com/post/curtain-up)',
 'Merry Christmas/Holidays 

In [16]:
# Resolve column conflict
if 'termy' in top_10_msgs.columns:
    top_10_msgs = top_10_msgs.drop(columns=['termy'])

# Merge without conflicts
top_10_msgs = top_10_msgs.merge(top_10_topics, on='category', how='left')

# Select desired columns
#result = top_10_msgs[['id_x', 'id_y', 'group_id', 'category', 'message_id', 'message', 'termy']]
result = top_10_msgs[['id', 'group_id', 'category', 'message_id', 'message', 'termy']]

# Save to CSV
result.to_csv('Final_Klaatch_messages.csv', index=False)

print("CSV file 'top_10_messages.csv' has been created.")


CSV file 'top_10_messages.csv' has been created.
